# Actividad 2026 · Unidad 2

**Atribución:** esta actividad adapta material del [Deep Reinforcement Learning Course de Hugging Face](https://huggingface.co/learn/deep-rl-course). El contenido original pertenece a sus autores y colaboradores.

**Traducción:** texto en español elaborado con asistencia de IA a partir del material original. Puede contener errores; ante una duda, consultá el notebook fuente.

[Abrir esta unidad en Colab](https://colab.research.google.com/drive/1UKQmi0aBdxTb-kjVqE5iGQeblG2Pr1Ds)

Adaptación del [notebook oficial](https://github.com/huggingface/deep-rl-class/blob/e9ae624cfbe6ab02dc548e7281f0eb67a0912ed3/notebooks/unit2/unit2.ipynb). Se entrenan y evalúan FrozenLake y Taxi-v4 en Colab. Las secciones de publicación y carga de modelos externos se excluyeron.


In [ ]:
%pip install -q gymnasium pygame numpy huggingface_hub imageio imageio-ffmpeg tqdm pyvirtualdisplay ipywidgets

# Unidad 2: Q Learning con FrozenLake-v1 ⛄ y Taxi-v4 🚕

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/thumbnail.jpg" alt="Portada de la unidad 2">

En este notebook vas a **programar desde cero un agente de Q Learning** para FrozenLake y Taxi, y experimentar con distintas configuraciones.


<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/envs.gif" alt="Environments"/>

### Entornos 🎮

- [FrozenLake-v1](https://gymnasium.farama.org/environments/toy_text/frozen_lake/)
- [Taxi-v4](https://gymnasium.farama.org/environments/toy_text/taxi/)

### Bibliotecas 📚

- Python y NumPy
- [Gymnasium](https://gymnasium.farama.org/)

Si encontrás un problema en el material original, podés [abrir un issue en su repositorio](https://github.com/huggingface/deep-rl-class/issues).


## Objetivos 🏆

Al terminar, vas a poder usar Gymnasium, implementar Q Learning desde cero y evaluar la política aprendida. La publicación en Hugging Face Hub queda fuera de esta adaptación.


## Material del curso de aprendizaje por refuerzos profundo de Hugging Face

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Ilustración del curso"/>


En el [curso gratuito de Hugging Face](https://huggingface.co/learn/deep-rl-course) se estudia aprendizaje por refuerzos profundo en teoría y práctica, se utilizan bibliotecas como Stable-Baselines3 y RL Baselines3 Zoo, y se entrenan agentes en distintos entornos.

Consultá el [programa original](https://simoninithomas.github.io/deep-rl-course) y su [comunidad de Discord](https://discord.gg/ydHrjt3WP5). El curso original también ofrece actividades y un certificado de finalización.


## Requisitos previos 🏗️

Antes de empezar, leé la [unidad 2 sobre Q Learning](https://huggingface.co/deep-rl-course/unit2/introduction).


## Repaso breve de Q Learning

**Q Learning** aprende una función de valor de acción, `Q(s,a)`. En entornos pequeños podemos guardarla en una **tabla Q** con una fila por estado y una columna por acción.

Para estimar el valor de una acción, consultamos la entrada correspondiente:

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-function-2.jpg" alt="Función Q" width="100%"/>

Cuando la estimación es suficientemente buena, podemos elegir en cada estado la acción con mayor valor y obtener así una política:

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/link-value-policy.jpg" alt="Relación entre valor y política" width="100%"/>

Al principio la tabla no contiene conocimiento útil: suele inicializarse en cero. A medida que el agente explora y actualiza sus valores, sus estimaciones mejoran.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/q-learning.jpeg" alt="Actualización de Q Learning" width="100%"/>

Este es el pseudocódigo de Q Learning:

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Pseudocódigo de Q Learning" width="100%"/>


# Programemos nuestro primer algoritmo de aprendizaje por refuerzos 🚀

## Instalá las dependencias 🔽

Las celdas siguientes instalan Gymnasium, NumPy y las bibliotecas de visualización necesarias. La adaptación evalúa las políticas localmente y grafica los resultados; no requiere publicar modelos en Hugging Face Hub.


Si Colab acaba de instalar una versión nueva de alguna biblioteca, puede pedir que reinicies el entorno de ejecución. Después del reinicio, continuá desde la celda de instalación.


**Ejercicio de preparación: usar la solución incluida más abajo.**


## Importá las bibliotecas 📦

Además de Gymnasium y NumPy, usamos `random` para la exploración ε-greedy e `imageio` para generar videos cuando se necesiten.


In [ ]:
import numpy as np
import gymnasium as gym
import random
import imageio
import os
import tqdm

import pickle
from tqdm.notebook import tqdm

Ya podemos programar Q Learning 🔥

# Parte 1: FrozenLake ⛄ sin deslizamiento

## Creá y explorá [FrozenLake ⛄](https://gymnasium.farama.org/environments/toy_text/frozen_lake/)

💡 Antes de usar un entorno, consultá su [documentación](https://gymnasium.farama.org/environments/toy_text/frozen_lake/).

El agente debe ir desde **S** hasta **G**, caminar sobre las casillas congeladas (**F**) y evitar los hoyos (**H**).

Hay mapas de `4x4` y `8x8`. Con `is_slippery=False`, cada acción mueve al agente en la dirección prevista; con `is_slippery=True`, el movimiento es estocástico y puede desviarse.


Empezamos con el mapa `4x4` sin deslizamiento. El parámetro `render_mode` controla la visualización. Para obtener cuadros como arreglos RGB y poder generar un video, usamos `render_mode="rgb_array"`. Consultá la [documentación de `render`](https://gymnasium.farama.org/api/env/#gymnasium.Env.render).


**Ejercicio de preparación: usar la solución incluida más abajo.**


### Solución

In [ ]:
# @title Solución (Mostrar código) { display-mode: "form" }
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="rgb_array")

También podés definir un mapa propio:

```python
desc = ["SFFF", "FHFH", "FFFH", "HFFG"]
gym.make("FrozenLake-v1", desc=desc, is_slippery=True)
```

Por ahora usaremos el mapa predeterminado.


### Veamos el entorno:

In [ ]:
# El entorno FrozenLake sin deslizamiento es determinista.
print("_____ESPACIO DE OBSERVACIÓN_____ \n")
print("Espacio de observación", env.observation_space)
print("Observación de ejemplo", env.observation_space.sample()) # Obtener una observación aleatoria

`Observation Space Shape Discrete(16)` indica que hay 16 estados. Cada observación es un entero que representa la posición: `fila * número_de_columnas + columna`, empezando desde cero. En el mapa `4x4`, la meta está en `3 * 4 + 3 = 15`.

Así se ve el estado `0`:

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/frozenlake.png" alt="FrozenLake"/>


In [ ]:
print("\n _____ESPACIO DE ACCIONES_____ \n")
print("Cantidad de acciones", env.action_space.n)
print("Acción de ejemplo", env.action_space.sample()) # Elegir una acción al azar

El agente tiene cuatro acciones 🎮:

- `0`: izquierda;
- `1`: abajo;
- `2`: derecha;
- `3`: arriba.

La recompensa es **+1 al llegar a la meta** y **0** al caer en un hoyo o pisar una casilla congelada.


## Creá e inicializá la tabla Q 🗄️

Este es el primer paso del pseudocódigo:

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q Learning" width="100%"/>

La tabla necesita una fila por estado y una columna por acción. Gymnasium permite obtener esas cantidades con `env.observation_space.n` y `env.action_space.n`, de modo que el código pueda adaptarse a otros entornos.


**Ejercicio: completar antes de ejecutar.** La solución aparece más adelante cuando está incluida en el original.

```python
state_space =
print("Hay ", state_space, " estados posibles")

action_space =
print("Hay ", action_space, " acciones posibles")
```


**Ejercicio: completar antes de ejecutar.** La solución aparece más adelante cuando está incluida en el original.

```python
# Creamos una tabla Q de tamaño (estados, acciones), inicializada en cero.
def initialize_q_table(state_space, action_space):
  Qtable =
  return Qtable
```


**Ejercicio de preparación: usar la solución incluida más abajo.**


### Solución

In [ ]:
# @title Solución (Mostrar código) { display-mode: "form" }
state_space = env.observation_space.n
print("Hay ", state_space, " estados posibles")

action_space = env.action_space.n
print("Hay ", action_space, " acciones posibles")

In [ ]:
# @title Solución (Mostrar código) { display-mode: "form" }
# Creamos una tabla Q de tamaño (estados, acciones), inicializada en cero.
def initialize_q_table(state_space, action_space):
  Qtable = np.zeros((state_space, action_space))
  return Qtable

In [ ]:
# @title Solución (Mostrar código) { display-mode: "form" }
Qtable_frozenlake = initialize_q_table(state_space, action_space)

## Definí la política greedy 🤖

Q Learning es un algoritmo **fuera de política**: durante el entrenamiento usamos una política con exploración para actuar, mientras que la actualización usa el mejor valor estimado del estado siguiente.

- Política ε-greedy: elige acciones durante el entrenamiento.
- Política greedy: elige la mejor acción según la tabla Q; también se usa para evaluar el agente entrenado.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/off-on-4.jpg" alt="Políticas de actuación y actualización" width="100%"/>


**Ejercicio: completar antes de ejecutar.** La solución aparece más adelante cuando está incluida en el original.

```python
def greedy_policy(Qtable, state):
  # Explotación: elegir la acción de mayor valor en este estado
  action =

  return action
```


#### Solución

In [ ]:
# @title Solución (Mostrar código) { display-mode: "form" }
def greedy_policy(Qtable, state):
  # Explotación: elegir la acción de mayor valor en este estado
  action = np.argmax(Qtable[state][:])

  return action

## Definí la política ε-greedy 🤖

Durante el entrenamiento combinamos explotación y exploración:

- Con probabilidad `1 − ε`, elegimos una acción de mayor valor según Q.
- Con probabilidad `ε`, elegimos una acción al azar.

A medida que avanza el entrenamiento, reducimos `ε` para explorar menos y aprovechar más lo aprendido.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-4.jpg" alt="Política épsilon-greedy" width="100%"/>


**Ejercicio: completar antes de ejecutar.** La solución aparece más adelante cuando está incluida en el original.

```python
def epsilon_greedy_policy(Qtable, state, epsilon):
  # Generar un número aleatorio entre 0 y 1
  random_num =
  # Si supera épsilon, explotar
  if random_num > epsilon:
    # Elegir la acción de mayor valor para este estado
    # np.argmax puede servir aquí
    action =
  # En caso contrario, explorar
  else:
    action = # Elegir una acción al azar

  return action
```


#### Solución

In [ ]:
# @title Solución (Mostrar código) { display-mode: "form" }
def epsilon_greedy_policy(Qtable, state, epsilon):
  # Generar un número aleatorio entre 0 y 1
  random_num = random.uniform(0,1)
  # Si supera épsilon, explotar
  if random_num > epsilon:
    # Elegir la acción de mayor valor para este estado
    # np.argmax puede servir aquí
    action = greedy_policy(Qtable, state)
  # En caso contrario, explorar
  else:
    action = env.action_space.sample()

  return action

## Definí los hiperparámetros ⚙️

La exploración debe durar lo suficiente para visitar los estados relevantes. Si `ε` disminuye demasiado rápido, el agente puede dejar de explorar antes de aprender una buena política.


In [ ]:
# Parámetros de entrenamiento
n_training_episodes = 10000  # Episodios de entrenamiento
learning_rate = 0.7          # Tasa de aprendizaje

# Parámetros de evaluación
n_eval_episodes = 100        # Episodios de evaluación

# Parámetros del entorno
env_id = "FrozenLake-v1"     # Nombre del entorno
max_steps = 99               # Pasos máximos por episodio
gamma = 0.95                 # Factor de descuento
eval_seed = []               # Semillas para la evaluación

# Parámetros de exploración
max_epsilon = 1.0             # Probabilidad inicial de exploración
min_epsilon = 0.05            # Probabilidad mínima de exploración
decay_rate = 0.0005            # Tasa de decaimiento de la exploración

## Construí el bucle de entrenamiento

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q Learning" width="100%"/>

En cada episodio:

1. Reducí `ε` y reiniciá el entorno.
2. Elegí una acción con la política ε-greedy.
3. Ejecutá la acción y observá el siguiente estado y la recompensa.
4. Actualizá `Q(s,a)` con la ecuación de Bellman: `Q(s,a) ← Q(s,a) + α [r + γ max Q(s',a') − Q(s,a)]`.
5. Continuá desde el nuevo estado hasta que termine el episodio o se alcance el límite de pasos.


**Ejercicio: completar antes de ejecutar.** La solución aparece más adelante cuando está incluida en el original.

```python
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
  for episode in tqdm(range(n_training_episodes)):
    # Reducir épsilon a medida que avanza el aprendizaje
    epsilon = min_epsilon + (max_epsilon - min_epsilon)*np.exp(-decay_rate*episode)
    # Reiniciar el entorno
    state, info = env.reset()
    step = 0
    terminated = False
    truncated = False

    # repeat
    for step in range(max_steps):
      # Elegir la acción con la política épsilon-greedy
      action =

      # Ejecutar la acción y observar recompensa y estado siguiente
      # Observar el nuevo estado y la recompensa
      new_state, reward, terminated, truncated, info =

      # Actualizar Q(s,a) con la ecuación de Bellman
      Qtable[state][action] =

      # Si termina o se trunca, cerrar el episodio
      if terminated or truncated:
        break

      # Continuar desde el nuevo estado
      state = new_state
  return Qtable
```


#### Solución

In [ ]:
# @title Solución (Mostrar código) { display-mode: "form" }
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable, history=None):
  # history es opcional para conservar el uso original de la función.
  if history is not None:
    history["rewards"] = []
    history["steps"] = []

  for episode in tqdm(range(n_training_episodes)):
    epsilon = min_epsilon + (max_epsilon - min_epsilon)*np.exp(-decay_rate*episode)
    state, info = env.reset()
    episode_reward = 0
    episode_steps = 0

    for step in range(max_steps):
      action = epsilon_greedy_policy(Qtable, state, epsilon)
      new_state, reward, terminated, truncated, info = env.step(action)
      episode_reward += reward
      episode_steps += 1

      # El valor futuro de un estado terminal es cero.
      future_value = 0 if terminated else np.max(Qtable[new_state])
      Qtable[state][action] += learning_rate * (
          reward + gamma * future_value - Qtable[state][action]
      )

      if terminated or truncated:
        break
      state = new_state

    if history is not None:
      history["rewards"].append(episode_reward)
      history["steps"].append(episode_steps)
  return Qtable

## Entrená el agente Q Learning 🏃

In [ ]:
history_frozenlake = {}
Qtable_frozenlake = train(
    n_training_episodes, min_epsilon, max_epsilon, decay_rate,
    env, max_steps, Qtable_frozenlake, history=history_frozenlake
)

## Curvas de recompensa, pasos y valores Q

Como en [`notebook_1_intro_rl.ipynb`](../notebooks/notebook_1_intro_rl.ipynb), observamos la recompensa y los pasos de **cada episodio de entrenamiento**. La media móvil ayuda a distinguir la tendencia de la variación entre episodios. En FrozenLake, la recompensa vale 0 o 1; en Taxi puede ser negativa. La curva mide el desempeño durante la exploración, no la evaluación final de la política *greedy*.

In [ ]:
import matplotlib.pyplot as plt


def plot_training(history, title, window):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, key, label in zip(
        axes, ("rewards", "steps"), ("Recompensa", "Pasos")
    ):
        values = np.asarray(history[key], dtype=float)
        episodes = np.arange(1, len(values) + 1)
        ax.plot(episodes, values, alpha=0.16, linewidth=0.7, label="Por episodio")
        if len(values) >= window:
            smoothed = np.convolve(values, np.ones(window) / window, mode="valid")
            ax.plot(episodes[window - 1:], smoothed, label=f"Media móvil ({window})")
        ax.set(xlabel="Episodio", ylabel=label)
        ax.legend()
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


plot_training(history_frozenlake, "FrozenLake: entrenamiento", window=100)

### Función de acción-valor en FrozenLake

Cada casilla muestra $V(s)=\max_a Q(s,a)$ y una flecha con la mejor acción según la tabla Q final. S es el inicio, G la meta y H un hoyo. Los estados terminales se marcan por separado: no tienen una acción siguiente. Si varias acciones tienen el mismo valor, `argmax` elige la primera; por eso una flecha en un estado sin explorar no prueba que se haya aprendido esa ruta.

In [ ]:
from matplotlib.colors import Normalize


def plot_frozenlake_values(q_table, environment):
    grid = environment.unwrapped.desc.astype(str)
    rows, cols = grid.shape
    values = np.max(q_table, axis=1).reshape(rows, cols)
    arrows = ("←", "↓", "→", "↑")
    fig, ax = plt.subplots(figsize=(6, 6))
    image = ax.imshow(values, cmap="YlGn", norm=Normalize(vmin=0, vmax=1))
    for row in range(rows):
        for col in range(cols):
            state = row * cols + col
            kind = grid[row, col]
            if kind in ("H", "G"):
                label = kind
            else:
                label = f"{kind if kind == 'S' else ''}{arrows[np.argmax(q_table[state])]}\n{values[row, col]:.2f}"
            ax.text(col, row, label, ha="center", va="center", fontsize=13)
    ax.set_xticks(range(cols))
    ax.set_yticks(range(rows))
    ax.set_title("FrozenLake: mejor acción y valor por estado")
    fig.colorbar(image, ax=ax, label="máx Q(s, a)")
    fig.tight_layout()
    plt.show()


plot_frozenlake_values(Qtable_frozenlake, env)

## Observá la tabla Q aprendida 👀

In [ ]:
Qtable_frozenlake

## Método de evaluación 📝

La siguiente función ejecuta episodios sin actualizar la tabla Q para medir el rendimiento de la política aprendida.


In [ ]:
def evaluate_agent(env, max_steps, n_eval_episodes, Q, seed):
  """
  Evalúa el agente en ``n_eval_episodes`` episodios y devuelve la media y desviación estándar de la recompensa.
  :param env: entorno de evaluación
  :param max_steps: pasos máximos por episodio
  :param n_eval_episodes: cantidad de episodios de evaluación
  :param Q: tabla Q
  :param seed: semillas de evaluación
  """
  episode_rewards = []
  for episode in tqdm(range(n_eval_episodes)):
    if seed:
      state, info = env.reset(seed=seed[episode])
    else:
      state, info = env.reset()
    step = 0
    truncated = False
    terminated = False
    total_rewards_ep = 0

    for step in range(max_steps):
      # Elegir la acción de mayor valor esperado para este estado
      action = greedy_policy(Q, state)
      new_state, reward, terminated, truncated, info = env.step(action)
      total_rewards_ep += reward

      if terminated or truncated:
        break
      state = new_state
    episode_rewards.append(total_rewards_ep)
  mean_reward = np.mean(episode_rewards)
  std_reward = np.std(episode_rewards)

  return mean_reward, std_reward

## Evaluá el agente Q Learning 📈

En el mapa `4x4` sin deslizamiento suele obtenerse una recompensa media cercana a **1**. Para un problema más difícil, probá la [versión resbaladiza](https://gymnasium.farama.org/environments/toy_text/frozen_lake/), que introduce resultados estocásticos.


In [ ]:
# Evaluar el agente
mean_reward, std_reward = evaluate_agent(env, max_steps, n_eval_episodes, Qtable_frozenlake, eval_seed)
print(f"Recompensa_media={mean_reward:.2f} +/- {std_reward:.2f}")

# Parte 2: Taxi-v4 🚖

## Creá y explorá [Taxi-v4 🚕](https://gymnasium.farama.org/environments/toy_text/taxi/)

💡 Consultá la [documentación del entorno](https://gymnasium.farama.org/environments/toy_text/taxi/).

En la grilla hay cuatro lugares designados: R (rojo), G (verde), Y (amarillo) y B (azul). El taxi empieza en una casilla aleatoria; debe buscar al pasajero, recogerlo, llevarlo a su destino y dejarlo allí. El episodio termina al completar la entrega.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/taxi.png" alt="Taxi"/>


In [ ]:
env = gym.make("Taxi-v4", render_mode="rgb_array")

Hay **500 estados discretos**: 25 posiciones del taxi, 5 ubicaciones posibles del pasajero (incluida la cabina) y 4 destinos.


In [ ]:
state_space = env.observation_space.n
print("Hay ", state_space, " estados posibles")

In [ ]:
action_space = env.action_space.n
print("Hay ", action_space, " acciones posibles")

El taxi dispone de seis acciones 🎮:

- `0`: sur; `1`: norte; `2`: este; `3`: oeste;
- `4`: recoger al pasajero; `5`: dejarlo.

La recompensa es **−1 por paso**, **+20 por entregar al pasajero** y **−10 por recogerlo o dejarlo de forma inválida**.


In [ ]:
# Crear una tabla Q de 500 estados por 6 acciones
Qtable_taxi = initialize_q_table(state_space, action_space)
print(Qtable_taxi)
print("Forma de la tabla Q: ", Qtable_taxi .shape)

## Definí los hiperparámetros ⚙️

⚠️ No modifiques `EVAL_SEED`: permite evaluar a todos los agentes desde las mismas posiciones iniciales.


In [ ]:
# Parámetros de entrenamiento
n_training_episodes = 25000   # Episodios de entrenamiento
learning_rate = 0.7           # Tasa de aprendizaje

# Parámetros de evaluación
n_eval_episodes = 100        # Episodios de evaluación

# NO MODIFICAR EVAL_SEED
eval_seed = [16,54,165,177,191,191,120,80,149,178,48,38,6,125,174,73,50,172,100,148,146,6,25,40,68,148,49,167,9,97,164,176,61,7,54,55,
 161,131,184,51,170,12,120,113,95,126,51,98,36,135,54,82,45,95,89,59,95,124,9,113,58,85,51,134,121,169,105,21,30,11,50,65,12,43,82,145,152,97,106,55,31,85,38,
 112,102,168,123,97,21,83,158,26,80,63,5,81,32,11,28,148] # Semillas que permiten evaluar a todos desde las mismas posiciones iniciales
                                                          # Cada semilla fija un estado inicial

# Parámetros del entorno
env_id = "Taxi-v4"           # Nombre del entorno
max_steps = 99               # Pasos máximos por episodio
gamma = 0.95                 # Factor de descuento

# Parámetros de exploración
max_epsilon = 1.0             # Probabilidad inicial de exploración
min_epsilon = 0.05           # Probabilidad mínima de exploración
decay_rate = 0.005            # Tasa de decaimiento de la exploración


## Entrená el agente Q Learning 🏃

In [ ]:
history_taxi = {}
Qtable_taxi = train(
    n_training_episodes, min_epsilon, max_epsilon, decay_rate,
    env, max_steps, Qtable_taxi, history=history_taxi
)
Qtable_taxi

### Curva de recompensa y valores Q de Taxi

El mapa de calor muestra los valores de las seis acciones para los primeros 25 estados codificados de Taxi. Una tabla de 500 estados es demasiado grande para leerla completa. Los valores negativos reflejan el costo de moverse o de hacer una acción inválida; la recompensa por episodio incluye toda la trayectoria.

In [ ]:
plot_training(history_taxi, "Taxi: entrenamiento", window=500)

fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(Qtable_taxi[:25], aspect="auto", cmap="RdYlGn")
ax.set_xticks(range(6), ["Sur", "Norte", "Este", "Oeste", "Recoger", "Dejar"])
ax.set_yticks(range(25))
ax.set(xlabel="Acción", ylabel="Estado codificado", title="Taxi: Q(s, a) para los estados 0–24")
fig.colorbar(image, ax=ax, label="Q(s, a)")
fig.tight_layout()
plt.show()

In [ ]:
# Evaluación local de Taxi-v4 (sin publicar el modelo)
mean_reward_taxi, std_reward_taxi = evaluate_agent(
    env, max_steps, n_eval_episodes, Qtable_taxi, eval_seed
)
print(f"Taxi-v4: recompensa media {mean_reward_taxi:.2f} ± {std_reward_taxi:.2f}")


## Para discutir
Compará las recompensas de FrozenLake y Taxi. ¿Qué cambia en el tamaño de la tabla Q y en la cantidad de episodios necesarios?
